# held out expression benchmark

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Render anatomical recovery and held-out expression MMD (Figure 4A–B)

In [ ]:
# Synthetic crescent | Results figure, FIRST PANEL
#
# Top:    Ground-truth anatomical coordinate of one synthetic tissue
# Middle: Zoetrope coordinate of the SAME tissue (absolute Spearman reported)
# Bottom: Held-out population gene-share MMD (four methods, bootstrap intervals)
#
# - Restores original barycentric MOSCOT MMD, not sampled-OT MMD.
# - Main MMD plot excludes static/no-change; the appendix retains it with CIs.
# - Recomputes the MOSCOT barycentric bootstrap CI with the SAME source/target
#   resampling scheme as the previous evaluation; never reuse the OT CI.
# - Reuses already saved bootstrap evaluations for the other four methods.
# - Exports STATIC-INCLUSIVE MMD and matched-cell RMSE to SEPARATE appendix figures.
# - No model fitting; no modification to frozen input or benchmark files.
# - SVG/PDF/PNG exact 3.15 x 5.50 inches; text as SVG outlines for PowerPoint.

from pathlib import Path
import os, json, hashlib, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.tri as mtri
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator, FuncFormatter
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, cdist

# --------------------------- CONFIGURATION ---------------------------
PROJECT = Path(os.getenv('STP_PROJECT',
    os.environ["STP_PROJECT"])).expanduser()
DATA = Path(os.getenv('CRESCENT_DATA_ROOT', str(PROJECT /
    'Experiments/simulation validation/Synthetic benchmarking/experiments/'
    'synthetic_developmental_crescent_v1'))).expanduser()
Z_FIG_DIR = Path(os.getenv('SYNTHETIC_Z_FIG_DIR', str(PROJECT /
    'experiments/synthetic_benchmark/publication_results/'
    'synthetic_anatomical_z_seed20261023_paper_compact'))).expanduser()
BASE = Path(os.getenv('SYNTHETIC_DIAGNOSTIC_DIR', str(PROJECT /
    'experiments/synthetic_benchmark/publication_results/'
    'heldout_expression_t2p5_seed20261023_decisive_diagnostics_v1'))).expanduser()
EVAL_DIR = Path(os.getenv('SYNTHETIC_RESULTS_UNCERTAINTY_OUT', str(BASE /
    'revised_distributional_MMD_and_matched_RMSE_v1'))).expanduser()
OUT = Path(os.getenv('SYNTHETIC_RESULTS_FIGURE_OUT', str(PROJECT /
    'experiments/synthetic_benchmark/publication_results/'
    'results_figure_panel_A_barycentric_restored'))).expanduser()
OUT.mkdir(parents=True, exist_ok=True)

SHOW_TISSUE = 7          # The ORIGINAL zero-based tissue ID; corresponds to Tissue 8 in Panel A.
DISPLAY_TISSUE = SHOW_TISSUE + 1
GENES = [f'G{i:02d}' for i in range(1, 49)]
METHODS = ['Spatial TI', 'STORIES', 'stVCR', 'moscot', 'Static']
DISPLAY = ['Zoetrope', 'STORIES', 'stVCR', 'moscot', 'Static']
# Keep all five for evaluation and appendix; show only competitors in the main figure.
MAIN_METHODS = METHODS[:4]
MAIN_DISPLAY = DISPLAY[:4]
FIG_W, FIG_H, DPI = 3.15, 5.50, 450
STEM = 'FIG_results_panel_A_tissue_coordinates_MMD_barycentric_four_methods_CI'
APPENDIX_STEM = 'FIG_appendix_matched_gene_share_RMSE_static_CI'
APPENDIX_MMD_STEM = 'FIG_appendix_heldout_gene_share_MMD_including_static_CI'
BOOT_SEED = 20261103

# ----------------------- DATA / PROVENANCE --------------------------
def require(p):
    p = Path(p)
    if not p.is_file():
        raise FileNotFoundError(f'Missing required artifact: {p}')
    return p

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

def gene_share(x):
    x = np.asarray(x, dtype=np.float64)
    assert x.ndim == 2 and x.shape[1] == 48 and np.isfinite(x).all()
    assert x.min() >= -1e-8
    x = np.maximum(x, 0)
    totals = x.sum(axis=1, keepdims=True)
    assert (totals > 0).all(), 'Zero-expression profile in gene shares'
    return x / totals

# Previously completed, leakage-controlled evaluation; do NOT use its moscot
# joint-OT MMD column for this panel.
summary_path = require(EVAL_DIR / 'heldout_MMD_RMSE_static_OT_CIs.csv')
boot_path = require(EVAL_DIR / 'bootstrap_draws.csv')
summary = pd.read_csv(summary_path).set_index('method').loc[METHODS]
boots = pd.read_csv(boot_path)
assert len(boots) == 200 and all(f'MMD_{k}' in boots.columns for k in METHODS)
assert all(f'RMSE_{k}' in boots.columns for k in METHODS)
N_BOOT = len(boots)

# Retrieve frozen coordinates (not model training or oracle construction).
metric_path = require(Z_FIG_DIR / 'per_tissue_absolute_spearman.csv')
tissue_path = require(Z_FIG_DIR / f'tissue_{SHOW_TISSUE:02d}_figure_data.csv.gz')
metrics = pd.read_csv(metric_path).sort_values('tissue_id')
tissue = pd.read_csv(tissue_path)
needed = {'x','y','z_true','z_stage1a','z_true_display','learned_z_display'}
assert needed.issubset(tissue.columns), needed - set(tissue.columns)
assert np.isfinite(tissue[list(needed)].to_numpy(float)).all()
rho = abs(float(spearmanr(tissue['z_stage1a'], tissue['z_true']).statistic))
reference_rho = float(metrics.loc[metrics.tissue_id.eq(SHOW_TISSUE),
                                  'abs_spearman'].iloc[0])
assert np.isclose(rho, reference_rho, atol=1e-9)
x = tissue['x'].to_numpy(float)
y = -tissue['y'].to_numpy(float)  # Publication vertical flip, as in original plot.
z_true = tissue['z_true_display'].to_numpy(float)
z_model = tissue['learned_z_display'].to_numpy(float)

# Use the already-frozen barycentric moscot prediction, NOT an OT sample.
manifest = json.loads(require(DATA / 'manifests/dataset_manifest.json').read_text())
assert manifest['dataset_id'] == 'synthetic_developmental_crescent_v1'
assert int(manifest['seed']) == 20261023 and manifest['status'] == 'frozen'
forecast_path = require(BASE / 'all_source_forecasts_FROZEN_BEFORE_TRUTH.npz')
with np.load(forecast_path, allow_pickle=False) as f:
    source_ids = f['cell_ids'].astype(str)
    assert f['genes'].astype(str).tolist() == GENES
    assert np.isclose(float(f['time_source']), 2/9)
    assert np.isclose(float(f['time_target']), 2.5/9)
    bary_share = gene_share(f['cal_moscot'])
assert len(source_ids) == len(set(source_ids)) == len(bary_share) == 2419

# Reproduce the EXACT MMD metric and the original bootstrap's source/target
# resampling for moscot's barycentric forecast (not its OT-sampling forecast).
count_path = require(DATA / 'train_only/public/counts.csv.gz')
count_frame = pd.read_csv(count_path)
assert count_frame.columns.tolist() == ['cell_id', *GENES]
count_frame['cell_id'] = count_frame['cell_id'].astype(str)
assert count_frame['cell_id'].is_unique
source_counts = count_frame.set_index('cell_id').loc[source_ids, GENES].to_numpy(float)
source_share = gene_share(source_counts)
target_path = require(DATA / 'evaluator_only/heldout/hidden/expected_rates.csv.gz')
target_frame = pd.read_csv(target_path)
assert target_frame.columns.tolist() == ['cell_id', *GENES]
target_share = gene_share(target_frame[GENES].to_numpy(float))

# Bandwidths: positive pairwise distances in 900 fixed source profiles.
rng_bw = np.random.default_rng(20261026)
subsample = source_share[rng_bw.choice(len(source_share), size=900, replace=False)]
dist = pdist(subsample)
median_distance = float(np.median(dist[dist > 0]))
assert median_distance > 0 and np.isfinite(median_distance)
widths = median_distance * np.array([0.5, 1.0, 2.0])

def gaussian_kernel(a, b, widths, block=320):
    ans = np.empty((len(a), len(b)), dtype=np.float64)
    for start in range(0, len(a), block):
        d2 = cdist(a[start:start+block], b, metric='sqeuclidean')
        kk = np.zeros_like(d2)
        for sigma in widths:
            kk += np.exp(-d2 / (2*sigma*sigma))
        ans[start:start+block] = kk / len(widths)
    return ans

print('Recomputing barycentric moscot MMD + conditional bootstrap ...')
Kxx = gaussian_kernel(bary_share, bary_share, widths)
Kxy = gaussian_kernel(bary_share, target_share, widths)
Kyy = gaussian_kernel(target_share, target_share, widths)
bary_mmd = float(np.sqrt(max(0.0, Kxx.mean() + Kyy.mean() - 2*Kxy.mean())))

# Same seed and same order of draws as the previous numerical evaluation cell.
rng = np.random.default_rng(BOOT_SEED + 999)
ns, nt = len(source_share), len(target_share)
WX = rng.multinomial(ns, np.full(ns, 1/ns), size=N_BOOT) / ns
WY = rng.multinomial(nt, np.full(nt, 1/nt), size=N_BOOT) / nt

def quadratic(weights_a, kernel, weights_b):
    return np.sum((weights_a @ kernel) * weights_b, axis=1)

bary_mmd_boot = np.sqrt(np.maximum(0.0,
    quadratic(WX, Kxx, WX) + quadratic(WY, Kyy, WY)
    - 2 * quadratic(WX, Kxy, WY)))
del Kxx, Kxy, Kyy

original_path = require(BASE / 'independent_target_population_MMD.csv')
original = pd.read_csv(original_path).set_index('method')
assert np.isclose(bary_mmd, float(original.loc['moscot','population_MMD']),
                  atol=5e-4), 'Barycentric moscot does not reproduce frozen MMD'
assert np.isclose(bary_mmd, 0.102602, atol=5e-4)

scores = {k: float(summary.loc[k,'MMD_gene_share']) for k in METHODS}
mmd_boot = {k: boots[f'MMD_{k}'].to_numpy(float) for k in METHODS}
scores['moscot'] = bary_mmd
mmd_boot['moscot'] = bary_mmd_boot
for k in METHODS[:4]:
    assert abs(scores[k] - float(original.loc[k,'population_MMD'])) < 5e-4
assert abs(scores['Static'] - 0.266476) < 5e-4

mmd_rows = []
for name, display in zip(METHODS, DISPLAY):
    low, high = np.quantile(mmd_boot[name], [0.025, 0.975])
    mmd_rows.append(dict(method=name, label=display,
                         MMD=scores[name], MMD_CI_low=float(low),
                         MMD_CI_high=float(high),
                         adapter='barycentric' if name=='moscot'
                                 else 'static_no_change' if name=='Static'
                                 else 'frozen_pipeline',
                         uncertainty='paired conditional bootstrap (frozen fits)'))
all_mmd_table = pd.DataFrame(mmd_rows)
all_mmd_table.to_csv(OUT / 'appendix_MMD_all_methods_including_static.csv', index=False)
main_table = all_mmd_table.loc[all_mmd_table['method'].isin(MAIN_METHODS)].copy()
main_table.to_csv(OUT / 'main_panel_MMD_four_methods_barycentric_bootstrap.csv', index=False)
print('\nMain-panel MMD (four methods; barycentric moscot):')
print(main_table.to_string(index=False, float_format=lambda v:f'{v:.6f}'))
print('\nStatic control retained in appendix:', f'{scores["Static"]:.6f}')

# ----------------------------- PLOT STYLE -----------------------------
available = {f.name for f in fm.fontManager.ttflist}
FONT = next((name for name in ['Arial','Liberation Sans','Arimo','Nimbus Sans','DejaVu Sans']
             if name in available), 'DejaVu Sans')
INK = '#23313B'; GRID = '#D2D7DC'; BLUE = '#4C78A8'; STATIC = '#A2A9B0'
mpl.rcParams.update({
    'font.family':FONT, 'font.size':10.5,
    'axes.labelsize':11.5, 'axes.titlesize':11.5,
    'xtick.labelsize':10, 'ytick.labelsize':10,
    'axes.linewidth':0.75, 'axes.edgecolor':INK,
    'axes.labelcolor':INK, 'xtick.color':INK, 'ytick.color':INK,
    'text.color':INK, 'pdf.fonttype':42, 'ps.fonttype':42,
    'svg.fonttype':'path', 'savefig.bbox':None,
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'savefig.facecolor':'white'})

# Use EXACT original tissue contouring rule; does not alter frozen data.
def masked_triangulation(x, y):
    tri = mtri.Triangulation(x, y)
    points = np.column_stack([x, y])
    v = tri.triangles
    edges = np.stack([
        np.linalg.norm(points[v[:,0]]-points[v[:,1]],axis=1),
        np.linalg.norm(points[v[:,1]]-points[v[:,2]],axis=1),
        np.linalg.norm(points[v[:,2]]-points[v[:,0]],axis=1)],axis=1)
    longest = edges.max(axis=1)
    threshold = max(1.8*np.median(longest), np.quantile(longest,0.94))
    tri.set_mask(longest > threshold)
    assert np.any(~tri.mask), 'No valid triangles to contour'
    return tri

TRI = masked_triangulation(x, y)
CONTOURS = np.linspace(0.12,0.88,5)
xpad, ypad = .03*np.ptp(x), .05*np.ptp(y)
XLIM=(float(x.min()-xpad),float(x.max()+xpad))
YLIM=(float(y.min()-ypad),float(y.max()+ypad))

fig = plt.figure(figsize=(FIG_W,FIG_H), facecolor='white')
# Geometry in PHYSICAL INCHES; labels are explicitly within the SVG viewport.
MAP_LEFT, MAP_W, MAP_H = .20, 2.22, .96
TRUE_BOTTOM, ZOETROPE_BOTTOM = 3.95, 2.75
CB_LEFT, CB_BOTTOM, CB_W, CB_H = 2.51, 2.82, .105, 2.02
BAR_LEFT, BAR_BOTTOM, BAR_W, BAR_H = .84, .84, 2.10, 1.35

def plot_tissue(bottom, expression, title):
    ax = fig.add_axes([MAP_LEFT/FIG_W, bottom/FIG_H,
                       MAP_W/FIG_W, MAP_H/FIG_H])
    ax.scatter(x,y,c=expression,cmap='viridis',vmin=0,vmax=1,
               s=3.3,linewidths=0,rasterized=True,zorder=2)
    ax.tricontour(TRI, expression, levels=CONTOURS,
                  colors=INK,linewidths=1.85,alpha=.82,zorder=3)
    ax.tricontour(TRI, expression, levels=CONTOURS,
                  colors='white',linewidths=1.05,alpha=.98,zorder=4)
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
    ax.set_aspect('equal',adjustable='box'); ax.set_axis_off()
    fig.text((MAP_LEFT+MAP_W/2)/FIG_W,(bottom+MAP_H+.14)/FIG_H,
             title,ha='center',va='center',fontsize=11.5,
             fontweight='bold',color=INK)
    return ax

fig.text(1.55/FIG_W,5.37/FIG_H,
         f'Synthetic tissue {DISPLAY_TISSUE}  ·  |ρ| = {rho:.3f}',
         ha='center',va='center',fontsize=11.2,fontweight='bold',color=INK)
ax_truth=plot_tissue(TRUE_BOTTOM,z_true,'Ground-truth')
ax_model=plot_tissue(ZOETROPE_BOTTOM,z_model,'Zoetrope')

# Shared colourbar; fixed-position title, rotated top-to-bottom (270 degrees).
cax=fig.add_axes([CB_LEFT/FIG_W,CB_BOTTOM/FIG_H,CB_W/FIG_W,CB_H/FIG_H])
cb=fig.colorbar(ScalarMappable(norm=Normalize(0,1),cmap='viridis'),cax=cax)
cb.set_ticks([0,.5,1]); cb.set_ticklabels(['0.0','0.5','1.0'])
cb.ax.tick_params(labelsize=10.5,length=2.8,width=.7,pad=1.5)
cb.outline.set_linewidth(.7)
fig.text(3.04/FIG_W,(CB_BOTTOM+CB_H/2)/FIG_H,
         'Common coordinate',rotation=270,ha='center',va='center',
         fontsize=10.5,fontweight='normal',color=INK)

# Population MMD: FOUR methods only. Static retained for the appendix.
ax=fig.add_axes([BAR_LEFT/FIG_W,BAR_BOTTOM/FIG_H,BAR_W/FIG_W,BAR_H/FIG_H])
xs_main=np.arange(len(MAIN_METHODS))
values=np.array([scores[k] for k in MAIN_METHODS])
lo=np.array([np.quantile(mmd_boot[k],.025) for k in MAIN_METHODS])
hi=np.array([np.quantile(mmd_boot[k],.975) for k in MAIN_METHODS])
yerr=np.vstack([np.maximum(0,values-lo), np.maximum(0,hi-values)])
ax.bar(xs_main,values,color=BLUE,width=.75,edgecolor='none',zorder=3)
ax.errorbar(xs_main,values,yerr=yerr,fmt='none',ecolor=INK,
            elinewidth=.85,capsize=2.1,capthick=.85,zorder=4)
ax.set_xlim(-.55,len(MAIN_METHODS)-.45)
ax.set_ylim(0, max(values.max(), hi.max())*1.30)
ax.set_xticks(xs_main,MAIN_DISPLAY,rotation=43,ha='right',va='top',
              rotation_mode='anchor',fontsize=9.4,fontweight='bold')
ax.yaxis.set_major_locator(MaxNLocator(4))
ax.yaxis.set_major_formatter(FuncFormatter(
    lambda v,p: '0' if abs(v)<1e-10 else f'{v:.3f}'))
ax.set_ylabel('Gene-share MMD',fontsize=11.5,labelpad=4)
ax.tick_params(axis='x',length=0,pad=1)
ax.tick_params(axis='y',length=3,width=.7,pad=2,labelsize=10)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y',color=GRID,alpha=.42,linewidth=.55)
ax.set_axisbelow(True)
for i, val in enumerate(values):
    ax.annotate(f'{val:.3f}',(i,hi[i]),xytext=(0,2.5),
                textcoords='offset points',ha='center',va='bottom',
                fontsize=8.2,color=INK)
fig.text(.50,2.45/FIG_H,'Held-out expression · MMD',
         ha='center',va='center',fontsize=11.2,fontweight='bold',color=INK)

# ----------------------- BOUNDS + EXPORT ----------------------------
def check_canvas(figure, width, height):
    figure.canvas.draw()
    box=figure.get_tightbbox(figure.canvas.get_renderer()).extents
    print('Artist bounds (inches):', [round(float(q),3) for q in box])
    tol=.005
    assert box[0]>=-tol and box[1]>=-tol and box[2]<=width+tol and box[3]<=height+tol, (
        f'Artist outside fixed SVG canvas: {box}; canvas={width} x {height} in')
    return box

check_canvas(fig,FIG_W,FIG_H)
for ext in ('svg','pdf','png'):
    path=OUT/f'{STEM}.{ext}'
    fig.savefig(path,dpi=DPI,bbox_inches=None,facecolor='white')
    print('SAVED:',path)
svg=ET.parse(OUT/f'{STEM}.svg').getroot()
print('Main SVG dimensions:',svg.attrib.get('width'),svg.attrib.get('height'))
plt.show()
plt.close(fig)

# ------------------- APPENDIX: STATIC MMD CONTROL ----------------------
# The main figure intentionally excludes the static bar, but its original
# result and corresponding 95% conditional bootstrap interval remain here.
MMD_FIG_W, MMD_FIG_H = 3.15, 2.75
fig_m = plt.figure(figsize=(MMD_FIG_W,MMD_FIG_H),facecolor='white')
ax_m=fig_m.add_axes([.78/MMD_FIG_W,.78/MMD_FIG_H,
                   2.10/MMD_FIG_W,1.43/MMD_FIG_H])
xs_all=np.arange(len(METHODS))
mm=np.array([scores[k] for k in METHODS])
mlow=np.array([np.quantile(mmd_boot[k], .025) for k in METHODS])
mhigh=np.array([np.quantile(mmd_boot[k], .975) for k in METHODS])
ax_m.bar(xs_all,mm,color=[BLUE]*4+[STATIC],width=.75,zorder=3)
ax_m.errorbar(xs_all,mm,
              yerr=np.vstack([np.maximum(0,mm-mlow),np.maximum(0,mhigh-mm)]),
              fmt='none',ecolor=INK,elinewidth=.85,capsize=2,capthick=.85,zorder=4)
ax_m.set_xlim(-.55,4.55);ax_m.set_ylim(0,max(mm.max(),mhigh.max())*1.3)
ax_m.set_xticks(xs_all,DISPLAY,rotation=43,ha='right',va='top',
                rotation_mode='anchor',fontsize=9.4,fontweight='bold')
ax_m.set_ylabel('Gene-share MMD',fontsize=11.5,labelpad=4)
ax_m.yaxis.set_major_locator(MaxNLocator(4))
ax_m.yaxis.set_major_formatter(FuncFormatter(
    lambda v,p: '0' if abs(v)<1e-10 else f'{v:.3f}'))
ax_m.spines[['top','right']].set_visible(False)
ax_m.tick_params(axis='x',length=0,pad=1)
ax_m.tick_params(axis='y',length=3,width=.7,pad=2,labelsize=10)
ax_m.grid(axis='y',color=GRID,alpha=.42,linewidth=.55)
ax_m.set_axisbelow(True)
for i,v in enumerate(mm):
    ax_m.annotate(f'{v:.3f}',(i,mhigh[i]),xytext=(0,2.5),
                  textcoords='offset points',ha='center',va='bottom',fontsize=8.2,color=INK)
fig_m.text(.50,2.57/MMD_FIG_H,'Held-out expression · MMD',
           ha='center',va='center',fontsize=11.2,fontweight='bold',color=INK)
check_canvas(fig_m,MMD_FIG_W,MMD_FIG_H)
for ext in ('svg','pdf','png'):
    path=OUT/f'{APPENDIX_MMD_STEM}.{ext}'
    fig_m.savefig(path,dpi=DPI,bbox_inches=None,facecolor='white')
    print('SAVED appendix MMD:',path)
plt.show()
plt.close(fig_m)

# ------------------- APPENDIX: MATCHED-CELL RMSE ----------------------
# Secondary supporting analysis is retained (not dropped), but exported as a
# separate 3.15 x 2.75-inch appendix panel. All values are from the existing
# 363 matched-cell, paired-source bootstrap; MOSCOT uses barycentric forecast.
RMSE_FIG_W, RMSE_FIG_H = 3.15,2.75
fig_s = plt.figure(figsize=(RMSE_FIG_W,RMSE_FIG_H),facecolor='white')
ax_s=fig_s.add_axes([.78/RMSE_FIG_W,.78/RMSE_FIG_H,
                   2.10/RMSE_FIG_W,1.43/RMSE_FIG_H])
rmse=np.array([float(summary.loc[k,'RMSE_gene_share'])*1e3 for k in METHODS])
rlow=np.array([float(summary.loc[k,'RMSE_CI_low'])*1e3 for k in METHODS])
rhigh=np.array([float(summary.loc[k,'RMSE_CI_high'])*1e3 for k in METHODS])
ax_s.bar(xs_all,rmse,color=[BLUE]*4+[STATIC],width=.75,zorder=3)
ax_s.errorbar(xs_all,rmse,yerr=np.vstack([np.maximum(0,rmse-rlow),np.maximum(0,rhigh-rmse)]),
              fmt='none',ecolor=INK,elinewidth=.85,capsize=2,capthick=.85,zorder=4)
ax_s.set_xlim(-.55,4.55); ax_s.set_ylim(0,rhigh.max()*1.3)
ax_s.set_xticks(xs_all,DISPLAY,rotation=43,ha='right',va='top',
                rotation_mode='anchor',fontsize=9.4,fontweight='bold')
ax_s.set_ylabel(r'Gene-share RMSE ($10^{-3}$)',fontsize=11.5,labelpad=4)
ax_s.yaxis.set_major_locator(MaxNLocator(4))
ax_s.yaxis.set_major_formatter(FuncFormatter(lambda v,p:f'{v:.1f}'))
ax_s.spines[['top','right']].set_visible(False)
ax_s.tick_params(axis='x',length=0,pad=1)
ax_s.tick_params(axis='y',length=3,width=.7,pad=2,labelsize=10)
ax_s.grid(axis='y',color=GRID,alpha=.42,linewidth=.55); ax_s.set_axisbelow(True)
for i,v in enumerate(rmse):
    ax_s.annotate(f'{v:.2f}',(i,rhigh[i]),xytext=(0,2.5),
                  textcoords='offset points',ha='center',va='bottom',fontsize=8.2,color=INK)
fig_s.text(.50,2.57/RMSE_FIG_H,'Matched-cell prediction · RMSE',
           ha='center',va='center',fontsize=11.2,fontweight='bold',color=INK)
check_canvas(fig_s,RMSE_FIG_W,RMSE_FIG_H)
for ext in ('svg','pdf','png'):
    path=OUT/f'{APPENDIX_STEM}.{ext}'
    fig_s.savefig(path,dpi=DPI,bbox_inches=None,facecolor='white')
    print('SAVED appendix:',path)
plt.show()
plt.close(fig_s)

# ---------------------------- RECEIPTS ------------------------------
main_receipt=dict(
    figure='results first panel: tissue Spearman + held-out population MMD',
    selected_tissue_id_zero_based=SHOW_TISSUE,display_tissue=DISPLAY_TISSUE,
    abs_spearman=rho,main_plot_methods=MAIN_DISPLAY,appendix_methods=DISPLAY,
    mmd={name:float(scores[name]) for name in METHODS},
    moscot_population_adapter='barycentric mean; frozen original pipeline',
    other_methods='frozen forecasts; static is observed source (appendix only)',
    uncertainty='200 conditional paired source/target empirical bootstrap replicates; fixed trained models',
    mmd_kernel='three Gaussian bandwidths from source-only median pairwise gene-share distance',
    mmd_bandwidths=widths.tolist(),
    source_forecasts_sha256=sha256(forecast_path),
    figure_inches=[FIG_W,FIG_H],appendix_rmse_figure=APPENDIX_STEM,
    appendix_static_MMD_figure=APPENDIX_MMD_STEM,
    note='moscot joint-coupling MMD sensitivity retained in prior evaluation outputs, not used in main plot')
(OUT/f'{STEM}_receipt.json').write_text(json.dumps(main_receipt,indent=2)+'\n')
print('\nFour-method main panel, static-inclusive appendix MMD, and appendix RMSE saved to:',OUT)